# COPD Exacerbation Risk Scoring

Derives per-patient COPD exacerbation risk scores from CPET (cardiopulmonary exercise
testing) data.  No ground-truth labels are required: risk is inferred from established
clinical thresholds (ATS/ERS guidelines) applied to extracted exercise biomarkers.

**Pipeline**
1. Load converted CSV files from `DATA_ROOT`
2. Extract CPET markers per patient (`fn.extract_copd_features`)
3. Score each patient against clinical thresholds (`fn.score_copd_risk`)
4. Visualise: radar charts, risk bar chart, PCA scatter
5. Export `output/copd_risk_summary.csv`

In [ ]:
import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import functions as fn

warnings.filterwarnings("ignore")

fn.setup_logging(log_file="pipeline.log", level=logging.INFO)
logger = logging.getLogger("copd_risk")

In [ ]:
# ── Configure paths ──────────────────────────────────────────────────────────
DATA_ROOT: str = "data"
OUTPUT_ROOT: str = "output"

output_base = Path(OUTPUT_ROOT)
copd_out = output_base / "copd_risk"
copd_out.mkdir(parents=True, exist_ok=True)

logger.info("COPD risk pipeline started. Data: %s  Output: %s", DATA_ROOT, copd_out)

## 1. Discover & load patient data

In [ ]:
patient_folders = fn.discover_patient_folders(DATA_ROOT)
print(f"Found {len(patient_folders)} patient folder(s):")
for pf in patient_folders:
    print(" -", pf.name)

## 2. Extract CPET features and score risk per patient

In [ ]:
records = []

for folder in patient_folders:
    patient_id = folder.name
    logger.info("Processing: %s", patient_id)

    csv_path = fn.find_csv_file(folder)
    if csv_path is None:
        logger.warning("  No CSV found for %s — skipping.", patient_id)
        continue

    try:
        info_df, telemetry_df = fn.load_telemetry(csv_path)
        meta = fn.parse_patient_meta(info_df)

        feats = fn.extract_copd_features(
            telemetry_df, weight_kg=meta.get("weight_kg")
        )
        scored = fn.score_copd_risk(feats)

        row = {"patient_id": patient_id, **meta, **scored.to_dict()}
        records.append(row)
        logger.info(
            "  %s → risk: %s  flags: %d",
            patient_id, scored["risk_score"], int(scored["n_flags"]),
        )
    except Exception as exc:
        logger.error("  FAILED for %s: %s", patient_id, exc, exc_info=True)

summary_df = pd.DataFrame(records)
print(f"\nProcessed {len(summary_df)} patient(s).")
if not summary_df.empty:
    display(summary_df[["patient_id", "age_years", "sex", "weight_kg",
                         "peak_vo2_per_kg", "ve_vco2_slope", "spo2_nadir",
                         "spo2_drop", "n_flags", "risk_score"]])

## 3. Visualisation

### 3a. Radar chart per patient

In [ ]:
feature_cols = [
    "peak_vo2_ml_min", "peak_vo2_per_kg", "ve_vco2_slope",
    "spo2_nadir", "spo2_drop", "bf_peak", "rer_peak",
    "o2_pulse_peak", "vt1_present",
]

for _, row in summary_df.iterrows():
    feats_series = row[feature_cols]
    fig = fn.plot_copd_radar(
        feats_series,
        patient_id=row["patient_id"],
        output_dir=copd_out,
    )
    plt.show()
    plt.close(fig)

### 3b. Risk score bar chart (all patients)

In [ ]:
if not summary_df.empty:
    risk_order = ["Low", "Moderate", "High"]
    colour_map = {"Low": "#2ecc71", "Moderate": "#f39c12", "High": "#e74c3c"}

    fig, ax = plt.subplots(figsize=(max(6, len(summary_df) * 0.8), 4))
    colours = [colour_map.get(r, "grey") for r in summary_df["risk_score"]]
    bars = ax.bar(summary_df["patient_id"], summary_df["n_flags"], color=colours)

    for bar, risk in zip(bars, summary_df["risk_score"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05,
            risk,
            ha="center", va="bottom", fontsize=9,
        )

    ax.set_xlabel("Patient")
    ax.set_ylabel("Number of risk flags")
    ax.set_title("COPD Exacerbation Risk — Flag Count per Patient")
    ax.set_ylim(0, 8)
    ax.axhline(y=3, color="#e74c3c", linestyle="--", alpha=0.5, label="High threshold (>2)")
    ax.axhline(y=1, color="#f39c12", linestyle="--", alpha=0.5, label="Moderate threshold (>0)")
    ax.legend(fontsize=8)
    plt.xticks(rotation=30, ha="right")
    fig.tight_layout()

    fname = copd_out / "copd_risk_bar.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    logger.info("Saved: %s", fname)
    plt.show()
    plt.close(fig)

### 3c. PCA scatter coloured by risk level

In [ ]:
if len(summary_df) >= 3:
    pca_cols = [c for c in feature_cols if c in summary_df.columns and c != "vt1_present"]
    pca_df = summary_df[pca_cols].apply(pd.to_numeric, errors="coerce")

    X_scaled, _, _ = fn.prepare_features(pca_df)
    pca_model, X_pca = fn.run_pca(X_scaled, n_components=3)

    fig = fn.plot_pca_scatter(
        X_pca,
        labels=summary_df["risk_score"].reset_index(drop=True),
        label_name="Risk",
        patient_id="all_patients",
        output_dir=copd_out,
    )
    plt.show()
    plt.close(fig)
else:
    print("Need ≥ 3 patients for PCA. Skipping.")

## 4. Export

In [ ]:
if not summary_df.empty:
    out_csv = output_base / "copd_risk_summary.csv"
    summary_df.to_csv(out_csv, index=False)
    logger.info("Saved summary: %s", out_csv)
    print(f"Saved: {out_csv}")
else:
    print("No data to export.")